# Figure 2 — single dye molecules on the Ximea camera

**Makes up:** Figure 2 b and c (a field of view of single ATTO 565 / ATTO 633 molecules, with a
photoelectron colour bar, and two single-step photobleaching traces with PSF zoom-ins) and e (ternary plot of eight dyes' relative
pixel QE, 1,000 molecules each). SI: `Colour_vs_Prediction` (measured vs predicted relative pixel
QE of all twelve dyes), `Single_Step_Photobleaching` (twelve photobleaching traces per dye), and
`ATTO_633_zoomins` / `ATTO_565_zoomins` (25 single-molecule PSFs of each, with Bayer keys).

**Not here:** Figure 2a is a schematic. Figure 2d (TetraSpeck bead precision) is made by
`Figure_2d_TetraspeckPrecision.ipynb`.

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/Single_Dye_Experiments/<dye>/`:
twelve dyes, each a 4 × 4 grid of positions (one TIFF of up to 1,250 frames at 100 ms, 1.4–2 GB,
plus one MicroManager metadata file each). The deposition holds raw data only, so this notebook fits it.

**Method:** fit each position (`fit_SM_data`, 12 px ROI, PSF wavelength = the dye's emission peak) →
HDBSCAN clustering of each dye's localisations into molecules (≥ 10 localisations, as in the
paper's Methods). Photobleaching: the raw 12 × 12 px photoelectron trace around each molecule must
show exactly one downward step (likelihood-ratio change-point detector). Shared code:
`_multifov.py` (download / fit / cluster) and `_single_dyes.py` (dyes, filters, bleaching screen).

## FAST_MODE

`FAST_MODE = True` (default) uses one position per dye (12 TIFFs of 1.4–2 GB, ~20 GB): ATTO 633's is
the position shown in the paper (`Pos002_003`), the others the first. Fitting takes ~2 min per
position; the download from EBI usually takes far longer (~15 min per GB-scale file at the speeds
seen). Every panel is made, from ~1/16 of the molecules. `FAST_MODE = False` uses all 192
positions (~300 GB; run in `tmux`/`screen`). Panels b and c then show each dye's best position.

---
## 0 — Imports

In [ ]:
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mpltern  # noqa: F401  (registers the "ternary" projection)
from scipy import ndimage

# _biostudies_download.py, _multifov.py and _single_dyes.py live beside this notebook.
sys.path.insert(0, str(Path.cwd().resolve()))
import _multifov as mf
import _single_dyes as sd

from pyS3M import IOFunctions
from pyS3M import HelperFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SpectralFunctions
from pyS3M import SR_Functions
from pyS3M import SM_extractionfunctions
from pyS3M import PlottingBase
from pyS3M.StepDetector import StepDetector

IO = IOFunctions.IO_Functions()
H_F = HelperFunctions.Helper_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
SM_E = SM_extractionfunctions.extract_SMs()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: one position per dye (~20 GB). False: all 192 positions (~300 GB, the published data).
REFIT = False      # True: fit again even if fits with the same files and settings exist.
FAST_POSITIONS = {"ATTO633": "Pos002_003"}   # FAST_MODE: the position Fig. 2c shows; others take the first

REPO_ROOT = Path.cwd().resolve().parents[1]   # notebooks/figures -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "Figure_2_SingleDyes"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_ROOT at a drive with room for the download. Each dye gets its own sub-folder,
# separately for each FAST_MODE setting: the fit reads every TIFF inside it.
DATA_ROOT = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

calib = dict(
    gain_map=IO.read_tiff(CALIB_DIR / "gain.tif"),
    offset_map=IO.read_tiff(CALIB_DIR / "offset.tif"),
    rqe=IO.read_tiff(CALIB_DIR / "rqe.tif"),
    read_noise=IO.read_tiff(CALIB_DIR / "readnoise.tif"),
    variance=IO.read_tiff(CALIB_DIR / "variance.tif"),
)

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_ROOT.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download and fit every dye

Per dye: download from BioStudies (with each position's metadata: the camera ROI the fit needs),
then fit each position separately. The PSF model's wavelength is the dye's emission peak.
Re-running is safe: complete downloads are skipped, and fits are reused unless the files or
settings change (or `REFIT = True`).

In [ ]:
dye_files, exposure_s = {}, {}
for dye, (spectrum_name, folder, _, _) in sd.DYES.items():
    fit_params = dict(sd.FIT_PARAMS, peak_wavelength=sd.emission_peak_um(S_F, spectrum_name, wavelength))
    print(f"\n=== {dye} (PSF wavelength {fit_params['peak_wavelength']} um)")
    dye_files[dye] = mf.prepare_acquisition(
        f"{sd.DYE_ROOT}/{folder}", DATA_ROOT / dye, FAST_MODE, SupRes_F, calib, smoothing_function, IO,
        fit_params, position=FAST_POSITIONS.get(dye), refit=REFIT,
    )
    exposure_s[dye] = IO.metadata_reader_imageJ(mf.metadata_for(mf.tif_for(dye_files[dye][0])),
                                                return_exposure=True)[4] / 1000

for dye, files in dye_files.items():
    print(f"{dye}: {len(files)} position(s), {exposure_s[dye] * 1000:.0f} ms frames")

---
## 3 — Cluster each dye's localisations into molecules

HDBSCAN, min. cluster size 10 localisations, after quality cuts (`sd.CLUSTER_PARAMS`: chi² < 3,
colour errors < 1 (no cut on the background errors), localisation error < 1 px, PSF sigma 75–250 nm, > 50 photoelectrons). A molecule's colour (`A_B`, `A_G`, `A_R`: relative signal on pixels 1,
2 and 3) is the inverse-variance weighted mean over its localisations; its photoelectrons are their sum.

In [ ]:
sm_db, sf_db = {}, {}
for dye, files in dye_files.items():
    sm_db[dye], sf_db[dye] = mf.cluster_sample(SM_E, files, DATA_ROOT / dye, dye, sd.CLUSTER_PARAMS)
    print(f"{dye}: {len(sm_db[dye]):,} molecules, median {sm_db[dye]['photoelectrons'].median():,.0f} photoelectrons")

---
## 4 — Figure 2e: eight dyes on a ternary plot

1,000 molecules of each dye with 7 × 10³–7 × 10⁴ detected photoelectrons, drawn at random (the published
2 × 10⁴–2 × 10⁵ `photons` × 0.35, the typical photoelectrons per `photon`, 0.25 (1 + A_G) on the Bayer mosaic). Full triangle (with the
dye legend and the zoomed region outlined in white), and a zoom on the red dyes. Axes: top =
pixel 3 (`A_R`), left = pixel 2 (`A_G`), right = pixel 1 (`A_B`), each in its pixel's colour.
(The analysis notebook labelled the top and right axes the other way round: pixel 1 and pixel 2.)
Styled as published by `sd.ternary_panel`: black triangle, dashed grid, triangle sides of 59.7 mm
(full) and 39.6 mm (zoom).

In [ ]:
PE_RANGE = (7e3, 7e4)   # published 2e4-2e5 `photons` (A_B + A_G + A_R) x 0.35
N_SHOWN = 1000
rng = np.random.default_rng(0)

ternary_dyes = [d for d, (_, _, _, colour) in sd.DYES.items() if colour is not None]
shown = {}
for dye in ternary_dyes:
    db = sm_db[dye][sm_db[dye]["photoelectrons"].between(*PE_RANGE)]
    shown[dye] = db.iloc[rng.choice(len(db), min(N_SHOWN, len(db)), replace=False)]
    print(f"{dye}: {len(db):,} of {len(sm_db[dye]):,} molecules in range, {len(shown[dye])} shown "
          f"(median {db['photoelectrons'].median():,.0f} photoelectrons)")

fig, ax = sd.ternary_panel(shown, legend=True, outline=sd.TERNARY_ZOOM["limits"], **sd.TERNARY_FULL)
plt.savefig(FIG_DIR / "ternary_rasterized.svg", dpi=600, format="svg")
plt.show()

# red zoom: pixel 3 0.6-0.8, pixel 2 0.2-0.4, pixel 1 0-0.2
fig, ax = sd.ternary_panel(shown, **sd.TERNARY_ZOOM)
plt.savefig(FIG_DIR / "red_zoom_tern_large.svg", dpi=600, format="svg")
plt.show()

---
## 5 — SI: measured vs predicted relative pixel QE, all twelve dyes

Histograms of every molecule's `A_B`, `A_G`, `A_R`; dashed lines: the prediction from the dye's
emission spectrum through that acquisition's filters and the objective, integrated against
each pixel's QE.

In [ ]:
# Same layout and style as the published SI figure: 4 × 3 dyes, probability density, one legend.
HIST_COLOURS = {"A_B": "blue", "A_G": "green", "A_R": "red"}
LINE_COLOURS = {"A_B": "blue", "A_G": "darkgreen", "A_R": "red"}
bins = np.linspace(0, 1, 101)
fig, axs = plt.subplots(4, 3, figsize=(6.69, 8), layout="constrained")
for ax, dye in zip(axs.ravel(), sd.DYES):
    db = sm_db[dye]
    predicted = sd.predicted_fingerprint(S_F, dye, wavelength, pixel_QYs)   # B, G, R
    for k, column in enumerate(["A_B", "A_G", "A_R"]):
        ax.hist(db[column], bins=bins, density=True, color=HIST_COLOURS[column], alpha=0.5,
                label=f"pixel {k + 1} QE")
    for k, column in enumerate(["A_B", "A_G", "A_R"]):
        ax.axvline(predicted[k], color=LINE_COLOURS[column], ls="--", lw=2,
                   label=f"predicted pixel {k + 1} QE")
    ax.set_xlim(0, 1)
    ax.set_xlabel("relative pixel QE", fontsize=8)
    ax.set_ylabel("probability density", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, ls="--", alpha=0.4)
    ax.set_title(sd.DYES[dye][0], fontsize=9)   # the published SI figure names each panel's dye
axs.ravel()[2].legend(loc="upper right", fontsize=6)
plt.savefig(FIG_DIR / "colour_vs_prediction.svg", dpi=600, format="svg")
plt.show()

---
## 6 — Screen every dye for single-step photobleaching

For each molecule seen in at least 25 frames, its 12 × 12 px photoelectron trace (first detection
to 50 frames after the last) must show exactly one downward step. Candidates are ranked by
on/off contrast. This reads each position's TIFF into memory once (1.4–3 GB), roughly 20 s per position.

In [ ]:
screen = sd.BleachScreen(StepDetector(win_size=sd.STEP_WIN_SIZE, alpha=sd.STEP_ALPHA, estimator="gaussian"),
                         IO, H_F, calib)
bleachers = {}
for dye in sd.DYES:
    bleachers[dye] = screen.screen(sf_db[dye], dye_files[dye])
    print(f"{dye}: {len(bleachers[dye]):,} single-step bleachers")

---
## 7 — SI: twelve single-step photobleaching traces per dye

The twelve highest-contrast traces of each dye (baseline-subtracted), with the step's segment means.

In [ ]:
N_ROWS, N_COLS = 3, 4
for dye in sd.DYES:
    fig, axs = plotter.two_column_plot(nrows=N_ROWS, ncols=N_COLS)
    axs = np.atleast_1d(axs).ravel()
    for i, ax in enumerate(axs):
        if i >= len(bleachers[dye]):
            ax.set_visible(False)
            continue
        cand = bleachers[dye][i]
        trace = cand["trace"] - cand["trace"].min()
        frames = np.arange(len(trace))
        ax.plot(frames, trace, color="black", lw=0.5, alpha=0.9, zorder=1)
        ax.plot(frames, screen.detector.segment_means(trace, cand["cps"]), color="#d40000", lw=1.1, zorder=2)
        ax.axvline(cand["cp"], color="#d40000", lw=0.8, ls="--", alpha=0.6, zorder=3)
        ax.set_xlim(frames[0], frames[-1])
        ax.set_ylim(bottom=0)
        ax.tick_params(labelsize=7, length=2, pad=1)
        if i % N_COLS == 0:
            ax.set_ylabel("intensity / pe", fontsize=8)
        if i // N_COLS == N_ROWS - 1:
            ax.set_xlabel("frame", fontsize=8)
    fig.savefig(FIG_DIR / f"{dye}_SI_Single_Dye_Photobleaching.svg", dpi=300, format="svg", bbox_inches="tight")
    plt.show()

---
## 8 — Figure 2b, c: ATTO 565 and ATTO 633 fields of view

As published (`sd.bleach_fov_panel`): the position with the two best, well-separated (≥ 10 px)
single-step bleachers that are clear of the insets and of the bottom strip. The field of view is
the maximum projection (photoelectrons, with a colour bar) over their on-times, 58.8 mm high.
Each molecule is boxed and joined to an inset (edged in the dye's colour; the second dashed)
holding its 10 × 10 px PSF (maximum projection over its own on-time; 300 nm scale bar) and its
photoelectron trace. For ATTO 633 each PSF is centred on the green-channel intensity centroid;
ATTO 565's positions needed no re-centring.
The 2 × 2 key in each PSF's top-left corner shows the colour filters of the pixels it covers, read
from the calibration's sensor masks at the zoom-in's position (ROI origin + crop).

In [ ]:
N_TRACES = 2
ZOOM_HALF = 11            # window searched for the zoom-in centre (px)
HALF = sd.ZOOM_PX // 2    # zoom-in half-width (10 x 10 px)


def zoom_crop(window, recentre):
    # the 10 x 10 px zoom-in, and its top-left pixel's (row, col) in the window
    if recentre:
        green = sCMOS.bayer_demosaic_stack(window)[0][:, :, 1]
        cy, cx = (int(round(v)) for v in ndimage.center_of_mass(green))
    else:
        cy, cx = window.shape[0] // 2, window.shape[1] // 2
    y0 = min(max(0, cy - HALF), window.shape[0] - 2 * HALF)
    x0 = min(max(0, cx - HALF), window.shape[1] - 2 * HALF)
    return window[y0:y0 + 2 * HALF, x0:x0 + 2 * HALF], (y0, x0)


# the calibration's full-sensor colour masks: each zoom-in's Bayer key is read at its sensor position
full_masks = SupRes_F.mask.get_masks(*calib["gain_map"].shape)


for dye in ["ATTO565", "ATTO633"]:
    layout = sd.FOV_PANELS[dye]
    by_fov = {}
    for cand in bleachers[dye]:
        fov = cand["fov_index"]
        _, _, w, h = IO.metadata_reader_imageJ(mf.metadata_for(mf.tif_for(dye_files[dye][fov])))
        if not sd.under_insets(cand["xc"], cand["yc"], w, h, layout["insets"]):
            by_fov.setdefault(fov, []).append(cand)
    eligible = {fov: sd.pick_spread(c, N_TRACES) for fov, c in by_fov.items()}
    eligible = {fov: c for fov, c in eligible.items() if len(c) == N_TRACES}
    if not eligible:
        print(f"{dye}: no position with {N_TRACES} well-separated bleachers clear of the insets")
        continue
    best_fov = max(eligible, key=lambda fov: sum(c["snr_ratio"] for c in eligible[fov]))
    top = sorted(eligible[best_fov], key=lambda c: c["yc"])   # upper molecule -> upper inset
    tif = mf.tif_for(dye_files[dye][best_fov])
    print(f"{dye}: {tif.name}, contrasts {[round(c['snr_ratio'], 1) for c in top]}")

    stack = sd.open_stack(tif)
    gain, offset, rqe = screen._maps_for(tif)
    roi_x, roi_y, _, _ = IO.metadata_reader_imageJ(mf.metadata_for(tif))   # camera ROI origin on the sensor
    n_frames, height, width = stack.shape
    f_lo, f_hi = min(c["f0"] for c in top), max(c["f0"] + c["cp"] for c in top)
    fov_img = screen.to_pe(stack[f_lo:f_hi].max(axis=0), gain, offset, rqe, 0, height, 0, width)
    for cand in top:
        xpx, ypx = int(round(cand["xc"])), int(round(cand["yc"]))
        y0, y1 = max(0, ypx - ZOOM_HALF), min(height, ypx + ZOOM_HALF)
        x0, x1 = max(0, xpx - ZOOM_HALF), min(width, xpx + ZOOM_HALF)
        window = screen.to_pe(stack[cand["f0"]:cand["f0"] + cand["cp"], y0:y1, x0:x1].max(axis=0),
                              gain, offset, rqe, y0, y1, x0, x1)
        cand["zoom"], (cy0, cx0) = zoom_crop(window, recentre=(dye != "ATTO565"))
        cand["zoom_origin"] = (roi_y + y0 + cy0, roi_x + x0 + cx0)   # sensor (row, col) of its top-left pixel
    del stack

    fig, ax = sd.bleach_fov_panel(fov_img, top, exposure_s[dye], layout["colour"], layout["insets"],
                                  mf.PIXEL_SIZE_NM, plotter, masks=full_masks)
    fig.savefig(FIG_DIR / f"{dye}_FOV_bleach.svg", dpi=600, format="svg")
    plt.show()

---
## 9 — SI: 25 single-molecule PSFs of ATTO 633 and ATTO 565

The `ATTO_633_zoomins` and `ATTO_565_zoomins` SI figures: 25 clear, isolated single-molecule PSFs from one
frame (frame 50: by then enough molecules have bleached that most PSFs stand alone) of the position used above
in FAST_MODE (ATTO 633 `Pos002_003`, ATTO 565 the first). Each is a 12 × 12 px crop (the fitting ROI) in
photoelectrons, with its own colour bar and a 300 nm scale bar. `_atto647n_cameras.pick_psfs` finds the peaks
of the Gaussian-smoothed frame, drops any with a neighbour above 20 % of their height in their crop and the
brightest 2 % (likely two molecules), and keeps the 25 brightest left. The 2 × 2 key in each crop's top-left
corner shows the colour filters of the pixels it covers, read from the calibration's sensor masks at the
crop's sensor position (camera ROI origin + crop).

In [ ]:
import _atto647n_cameras as cams

ZOOM_FRAME = 50      # frame of the position's stack the PSFs are taken from
N_PSFS = 25          # 5 x 5 grid
PSF_HALF = sd.FIT_PARAMS["ROI_size"] // 2   # 12 x 12 px crops
sensor_type = PlottingBase.bayer_pixel_type(full_masks)   # 0/1/2 = B/G/R filter of every sensor pixel


def position_tif(dye):
    # the position used in FAST_MODE (FAST_POSITIONS, else the first)
    tifs = [mf.tif_for(h5) for h5 in dye_files[dye]]
    pos = FAST_POSITIONS.get(dye)
    return next((t for t in tifs if pos is not None and pos in t.name), tifs[0])


for dye in ["ATTO633", "ATTO565"]:
    tif = position_tif(dye)
    roi_x, roi_y, width, height = IO.metadata_reader_imageJ(mf.metadata_for(tif))   # camera ROI on the sensor
    gain, offset, rqe = screen._maps_for(tif)
    frame_pe = screen.to_pe(sd.read_frame(tif, ZOOM_FRAME), gain, offset, rqe, 0, height, 0, width)
    centres = cams.pick_psfs(frame_pe, N_PSFS, PSF_HALF)
    crops = [frame_pe[r - PSF_HALF:r + PSF_HALF, c - PSF_HALF:c + PSF_HALF] for r, c in centres]
    types = [sensor_type[roi_y + r - PSF_HALF:roi_y + r + PSF_HALF, roi_x + c - PSF_HALF:roi_x + c + PSF_HALF]
             for r, c in centres]
    print(f"{dye}: {tif.name}, frame {ZOOM_FRAME}; peak photoelectrons "
          f"{min(c.max() for c in crops):.0f}-{max(c.max() for c in crops):.0f}")
    fig, axs = cams.plot_zoom_ins(plotter, crops, types, mf.PIXEL_SIZE_NM)
    fig.savefig(FIG_DIR / f"{dye}_ZoomIns.svg", dpi=600, format="svg")
    plt.show()
